# dsc-co: before / after demo

run the three baseline policies on a tier-2 scenario, plot terminal reward and network-state snapshots.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.policies import zero_op_rollout, greedy_rollout, optimal_replay_rollout

SEED = 42
TIER = 2
rollouts = {
    'zero_op': zero_op_rollout(seed=SEED, difficulty=TIER),
    'greedy': greedy_rollout(seed=SEED, difficulty=TIER),
    'optimal_replay': optimal_replay_rollout(seed=SEED, difficulty=TIER),
}
for name, r in rollouts.items():
    print(f'{name:16s} agent={r.agent_cost:10.1f}  opt={r.optimal_cost:10.1f}  gap={r.gap:.3f}  terminal={r.terminal:.3f}')

In [ ]:
import matplotlib.pyplot as plt

names = list(rollouts.keys())
terminals = [rollouts[n].terminal for n in names]
gaps = [min(rollouts[n].gap, 4.0) for n in names]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(names, terminals, color=['tomato','steelblue','seagreen'])
axes[0].set_ylim(0, 1.0)
axes[0].set_title(f'terminal reward (tier {TIER}, seed {SEED})')
axes[0].grid(True, axis='y', alpha=0.3)
axes[1].bar(names, gaps, color=['tomato','steelblue','seagreen'])
axes[1].set_title('optimality gap (capped at 4)')
axes[1].grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for name, color in [('zero_op','tomato'), ('greedy','steelblue'), ('optimal_replay','seagreen')]:
    traj = rollouts[name].trajectory
    steps = list(range(len(traj)))
    costs = [x.get('agent_cost', 0.0) for x in traj]
    ax.plot(steps, costs, label=name, color=color, linewidth=1.5)
ax.set_xlabel('env step')
ax.set_ylabel('cumulative agent cost')
ax.set_title('cumulative cost over rollout')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## loading a trained grpo lora

after running `notebooks/train_kaggle.ipynb`, load the saved adapter here and rerun the rollout cell to compare the trained policy against the baselines.

```python
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained('path/to/grpo_dsc_co/final', max_seq_length=8192, load_in_4bit=True, fast_inference=True)
FastLanguageModel.for_inference(model)
```